# Moving In Next Door

Day 13 · Interactions. Where America's new batteries and solar are being built, measured against the big gas plants already on the grid.
Run `python scripts\fetch_week3.py day13` first (it makes the two EIA and USGS files this notebook reads).

## 1 · Settings
Change the look here. Everything below reads from these.

In [ ]:
DAY      = 13
PROMPT   = "Interactions"
SLUG     = "interactions-batteries-gas"
NOTEBOOK = "day13_interactions_batteries_gas.ipynb"
TITLE    = "Moving In Next Door"
CREDITS  = ("Generators: U.S. Energy Information Administration, EIA-860M (September 2026). "
            "Solar footprints: USGS, U.S. Large-Scale Solar Photovoltaic Database (USPVDB). "
            "State boundaries: US Census Bureau.")

NEAR_KM     = 10          # "next door": within this distance of a large gas plant
GAS_MIN_MW  = 50          # a "large" gas plant: fossil natural gas, at least this many MW
NEW_FROM    = 2020        # "new": built this year or later
CLOSEUP     = "Red Hawk"  # the gas plant in the close-up (EIA plant name)
CLOSEUP_KM  = (29, 21.8)  # close-up window, width x height in km (keeps the 10 km circle whole)

LOOK = "paper"            # "paper", "night" or "riso"
LOOKS = {
    #           page       ink        soft       land       gas        battery    battery far  solar      solar old
    "paper": ("#f3f1ea", "#1f2428", "#7c858b", "#e4e1d7", "#1f2428", "#5b3df5", "#c9c3e6", "#f0b323", "#f5dc9a"),
    "night": ("#0f1316", "#eef1f2", "#8a959b", "#1b2226", "#ff8a3d", "#5ef0d2", "#2e4a47", "#ffd34d", "#5c5232"),
    "riso":  ("#f6efe6", "#1d3b8b", "#6f7fae", "#ebe1d3", "#1d3b8b", "#ff4f79", "#f4c1cc", "#ffb000", "#f7dca0"),
}
PAGE, INK, SOFT, LAND, GAS, BATT, BATT_FAR, SOLAR, SOLAR_OLD = LOOKS[LOOK]

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.spatial import cKDTree
from mapkit import ai, poster, publish
from mapkit.paths import PROCESSED, EQUAL_AREA

gens = pd.read_csv(PROCESSED / "eia860m_generators.csv")
farms = gpd.read_file(PROCESSED / "uspvdb_footprints.geojson")
states = ai.load_states()

LOWER48 = [s for s in states.state_abb if s not in ("AK", "HI", "PR", "VI", "GU", "AS", "MP")]
states = states[states.state_abb.isin(LOWER48)].to_crs(EQUAL_AREA)
gens = gens[gens.state.isin(LOWER48) & ~gens.status.str.startswith("(OS)")]   # drop units retired for good
print(f"{len(gens):,} generators in the Lower 48 · {len(farms):,} mapped solar farms")

## 3 · Analysis, every number on the map comes from here

In [ ]:
def points(df):
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(EQUAL_AREA)

# Large gas plants: fossil natural gas only (code NG, so no landfill gas or biogas), summed per plant
gas = (gens[(gens.status_group == "operating") & (gens.source == "NG")]
       .groupby("plant_id").agg(plant=("plant", "first"), mw=("mw", "sum"), lat=("lat", "first"), lon=("lon", "first"))
       .reset_index())
gas = points(gas[gas.mw >= GAS_MIN_MW])

# New clean plants: solar, wind and batteries built since NEW_FROM, one row per plant and fuel
new = gens[(gens.status_group == "operating") & (gens.year >= NEW_FROM) & gens.fuel.isin(["Solar", "Wind", "Storage"])]
new = points(new.groupby(["plant_id", "fuel"]).agg(plant=("plant", "first"), mw=("mw", "sum"),
                                                   lat=("lat", "first"), lon=("lon", "first")).reset_index())

# Distance from every new plant to the nearest large gas plant
tree = cKDTree(np.c_[gas.geometry.x, gas.geometry.y])
dist, _ = tree.query(np.c_[new.geometry.x, new.geometry.y])
new["km_to_gas"] = dist / 1000
new["next_door"] = new.km_to_gas <= NEAR_KM

# The fair comparison: how much of the Lower 48 lies that close to a large gas plant at all?
land = states.union_all()
rings = gas.buffer(NEAR_KM * 1000).union_all().intersection(land)
LAND_SHARE = rings.area / land.area * 100

SHARE = {f: d.mw[d.next_door].sum() / d.mw.sum() * 100 for f, d in new.groupby("fuel")}
MW = {f: d.mw.sum() for f, d in new.groupby("fuel")}
print(f"{len(gas):,} large gas plants, {gas.mw.sum()/1000:,.0f} GW")
print(f"Land within {NEAR_KM} km of one: {LAND_SHARE:.1f}% of the Lower 48")
for f in ("Storage", "Solar", "Wind"):
    print(f"  {f:8s} built {NEW_FROM}+: {MW[f]/1000:5.1f} GW, {SHARE[f]:4.1f}% of it within {NEAR_KM} km")
print(f"Batteries are {SHARE['Storage'] / LAND_SHARE:.1f}x more likely to sit next door than the land itself")

# Solar land use, straight from the USGS footprints
built_new = farms[farms.p_year >= NEW_FROM]
ACRES_PER_MW = built_new.p_area.sum() / 4046.856 / built_new.p_cap_ac.sum()
print(f"Solar farms built {NEW_FROM}+: {ACRES_PER_MW:.1f} acres per MW (AC), over {len(built_new):,} farms of 20 MW or more")

# The close-up
hub = gas[gas.plant == CLOSEUP].iloc[0]
near_hub = new[new.distance(hub.geometry) <= NEAR_KM * 1000]
farms_ea = farms.to_crs(EQUAL_AREA)
hub_farms = farms_ea[farms_ea.distance(hub.geometry) <= NEAR_KM * 1000]
HUB = dict(gas_mw=hub.mw,
           solar_mw=near_hub.mw[near_hub.fuel == "Solar"].sum(),
           batt_mw=near_hub.mw[near_hub.fuel == "Storage"].sum(),
           solar_acres=hub_farms[hub_farms.p_year >= NEW_FROM].p_area.sum() / 4046.856)
print(f"{CLOSEUP}: {HUB['gas_mw']:,.0f} MW gas. Within {NEAR_KM} km since {NEW_FROM}: "
      f"{HUB['solar_mw']:,.0f} MW solar, {HUB['batt_mw']:,.0f} MW batteries, {HUB['solar_acres']:,.0f} acres of new solar")

## 4 · Map

In [ ]:
fig, ax = poster.page(PAGE)
ax.set_position([0.03, 0.405, 0.94, 0.375])              # the national map
SERIF, SANS, SANS_B = poster.DM_SERIF, poster.BARLOW, poster.BARLOW_BOLD
F = poster.font

states.plot(ax=ax, color=LAND, edgecolor=PAGE, lw=0.6, zorder=1)
batt = new[new.fuel == "Storage"]
size = lambda mw, k: np.sqrt(mw) * k                     # area of each mark follows its MW
far, near = batt[~batt.next_door], batt[batt.next_door]
ax.scatter(far.geometry.x, far.geometry.y, s=size(far.mw, 1.1), color=BATT_FAR, lw=0, zorder=2)
ax.scatter(gas.geometry.x, gas.geometry.y, s=size(gas.mw, 1.1), facecolor="none", edgecolor=GAS, lw=0.55, alpha=0.75, zorder=3)
ax.scatter(near.geometry.x, near.geometry.y, s=size(near.mw, 1.1), color=BATT, lw=0, alpha=0.95, zorder=4)
ax.set_axis_off()
b = states.total_bounds
ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_aspect("equal")

# where the close-up is
cx, cy = hub.geometry.x, hub.geometry.y
ax.add_patch(plt.Rectangle((cx - 60e3, cy - 60e3), 120e3, 120e3, fill=False, ec=INK, lw=0.9, zorder=6))

# --- title block
fig.text(0.06, 0.957, TITLE, color=INK, fontproperties=F(SERIF, 56), va="top")
fig.text(0.06, 0.878, f"WITHIN {NEAR_KM} KM OF A LARGE GAS PLANT", color=SOFT, fontproperties=F(SANS_B, 13), va="top")
fig.text(0.06, 0.857, f"{LAND_SHARE:.0f}%", color=INK, fontproperties=F(SERIF, 42), va="top")
fig.text(0.06, 0.812, "of the land", color=SOFT, fontproperties=F(SANS, 15), va="top")
fig.text(0.29, 0.857, f"{SHARE['Storage']:.0f}%", color=BATT, fontproperties=F(SERIF, 42), va="top")
fig.text(0.29, 0.812, f"of the battery capacity built since {NEW_FROM}", color=SOFT, fontproperties=F(SANS, 15), va="top")

# --- legend, bottom right of the national map
lx, ly = 0.065, 0.452
for i, (lab, kw) in enumerate([("Large gas plant", dict(facecolor="none", edgecolor=GAS, lw=0.9)),
                               (f"New battery, next door", dict(color=BATT)),
                               (f"New battery, farther away", dict(color=BATT_FAR))]):
    y = ly - i * 0.022
    fig.add_artist(plt.matplotlib.patches.Circle((lx, y + 0.004), 0.0055, transform=fig.transFigure, **kw))
    fig.text(lx + 0.016, y, lab, color=INK, fontproperties=F(SANS, 12.5), va="bottom")

# --- close-up
ax2 = fig.add_axes([0.06, 0.085, 0.50, 0.30], facecolor=PAGE)
w, h = CLOSEUP_KM[0] * 1000 / 2, CLOSEUP_KM[1] * 1000 / 2
view = (cx - w, cy - h, cx + w, cy + h)
farms_ea.cx[view[0]:view[2], view[1]:view[3]].pipe(lambda d: d[d.p_year < NEW_FROM]).plot(ax=ax2, color=SOLAR_OLD, lw=0, zorder=1)
farms_ea.cx[view[0]:view[2], view[1]:view[3]].pipe(lambda d: d[d.p_year >= NEW_FROM]).plot(ax=ax2, color=SOLAR, lw=0, zorder=2)
ax2.add_patch(Circle((cx, cy), NEAR_KM * 1000, fill=False, ec=INK, lw=0.8, ls=(0, (4, 3)), zorder=3))
g2 = gas.cx[view[0]:view[2], view[1]:view[3]].copy()
g2["spot"] = g2.lat.round(3).astype(str) + g2.lon.round(3).astype(str)       # Mesquite's two blocks share one site
g2 = g2.dissolve("spot", aggfunc={"plant": "first", "mw": "sum"})
g2["geometry"] = g2.geometry.centroid
ax2.scatter(g2.geometry.x, g2.geometry.y, s=size(g2.mw, 2.2), facecolor=PAGE, edgecolor=GAS, lw=1.3, zorder=5)
b2 = batt.cx[view[0]:view[2], view[1]:view[3]]
ax2.scatter(b2.geometry.x, b2.geometry.y, s=size(b2.mw, 6), color=BATT, edgecolor=PAGE, lw=0.8, zorder=6)
LABEL_ABOVE = ("Mesquite",)                                 # labels that go above their ring, so neighbours don't collide
for _, r in g2.iterrows():                                # name every gas plant in the window
    main = r.plant == CLOSEUP
    name = r.plant.replace(" Generating Station Block 1", "").replace(" Generating Station Block 2", "").replace(" Energy Facility", "")
    up = name in LABEL_ABOVE
    poster.halo_text(ax2, r.geometry.x, r.geometry.y + (1100 if up else -1100), name, color=INK if main else SOFT,
                     path=SANS_B if main else SANS, size=12 if main else 10.5, halo=PAGE, halo_width=3,
                     ha="center", va="bottom" if up else "top", zorder=7)
nuke = gens[(gens.fuel == "Nuclear")].groupby("plant").agg(lat=("lat", "first"), lon=("lon", "first"), mw=("mw", "sum"))
nuke = points(nuke.reset_index()).cx[view[0]:view[2], view[1]:view[3]]
ax2.scatter(nuke.geometry.x, nuke.geometry.y, s=60, marker="s", facecolor=PAGE, edgecolor=SOFT, lw=1.1, zorder=5)
for _, r in nuke.iterrows():
    ax2.text(r.geometry.x, r.geometry.y + 900, f"{r.plant} nuclear", color=SOFT, ha="center", va="bottom",
             fontproperties=F(SANS, 10.5), zorder=7)
ax2.set_xlim(view[0], view[2]); ax2.set_ylim(view[1], view[3]); ax2.set_aspect("equal"); ax2.set_axis_off()
ax2.plot([view[0] + 800, view[0] + 5800], [view[1] + 900] * 2, color=INK, lw=1.4)
ax2.text(view[0] + 3300, view[1] + 1300, "5 km", color=INK, fontproperties=F(SANS, 10.5), ha="center", va="bottom")
for i, (lab, col) in enumerate([(f"Solar since {NEW_FROM}", SOLAR), (f"Solar before {NEW_FROM}", SOLAR_OLD)]):
    yy = view[3] - 900 - i * 1500
    ax2.add_patch(plt.Rectangle((view[0] + 700, yy - 450), 1100, 900, color=col, lw=0))
    ax2.text(view[0] + 2200, yy, lab, color=INK, fontproperties=F(SANS, 10.5), va="center")

# close-up notes, to the right
nx = 0.60
fig.text(nx, 0.375, f"{CLOSEUP.upper()}, ARIZONA", color=SOFT, fontproperties=F(SANS_B, 12.5), va="top")
notes = [(f"{HUB['gas_mw']:,.0f} MW", f"of gas at {CLOSEUP}", GAS),
         (f"{HUB['solar_mw']:,.0f} MW", f"of solar since {NEW_FROM}, within {NEAR_KM} km", SOLAR if LOOK != "paper" else "#c98a00"),
         (f"{HUB['batt_mw']:,.0f} MW", f"of batteries, same {NEAR_KM} km", BATT),
         (f"{HUB['solar_acres']:,.0f}", "acres of new solar arrays", INK)]
for i, (big, small, col) in enumerate(notes):
    y = 0.345 - i * 0.066
    fig.text(nx, y, big, color=col, fontproperties=F(SERIF, 27), va="top")
    fig.text(nx, y - 0.032, small, color=SOFT, fontproperties=F(SANS, 12.5), va="top")

poster.credit(fig, "EIA-860M generators (Sept 2026) · USGS solar footprints (USPVDB) · US Census\n"
                   f"Large gas plant: fossil gas, {GAS_MIN_MW} MW or more. Footprints shown are farms of 20 MW or more.",
              x=0.06, y=0.022, color=SOFT, path=SANS, name_path=SANS_B, name_color=INK)
plt.show()

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE,
             subtitle=f"{LAND_SHARE:.0f}% of the land, {SHARE['Storage']:.0f}% of new battery capacity, within {NEAR_KM} km of a large gas plant",
             series="ai", tool="Python", notebook=NOTEBOOK, credits=CREDITS)